Step 1 — Load the Dataset

In [1]:
import pandas as pd

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

movies = movies.merge(credits, on='title')

movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Why merge datasets?

Because one dataset has: movie metadata

The other has: cast and crew

Combining them improves recommendation quality.

Step 2 — Select Important Columns

We keep only useful features.

In [2]:
movies = movies[[
    "title",
    "genres",
    "keywords",
    "overview",
    "cast",
    "crew"
]]

Why remove other columns?

Many columns are:
budget
revenue
homepage
status
These do not help recommendation.

This step is called feature selection.

Step 3 — Handle Missing Data

In [3]:
movies.dropna(inplace=True)

Missing data causes:
vectorization errors And reduces model quality.

Step 4 — Convert JSON Columns

In [4]:
import ast
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i["name"])
    return L

In [5]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

Step 5 — Combine All Features

Now create the tags column.

In [6]:
movies["tags"] = movies["overview"] + " " + movies["genres"].astype(str) + " " + movies["keywords"].astype(str)

In [7]:
movies["tags"] = movies["tags"].apply(lambda x: x.lower())

Step 6 — Vectorization

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words="english")

vectors = cv.fit_transform(movies["tags"]).toarray()

Step 7 — Similarity Matrix

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

Step 8 - Save the Model
Computing similarity every time is slow, so we save it.

In [13]:
import pickle

pickle.dump(movies, open("movies.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))

So the web app can load it instantly.

Step 9 — Build a Web App (Streamlit)

In [14]:
pip install streamlit

   ---------------------------------------- 0.0/413.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/413.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/413.7 kB ? eta -:--:--
    --------------------------------------- 10.2/413.7 kB ? eta -:--:--
    --------------------------------------- 10.2/413.7 kB ? eta -:--:--
   - ------------------------------------- 20.5/413.7 kB 165.2 kB/s eta 0:00:03
   --- ----------------------------------- 41.0/413.7 kB 245.8 kB/s eta 0:00:02
   ------- ------------------------------- 81.9/413.7 kB 416.7 kB/s eta 0:00:01
   -------- ------------------------------ 92.2/413.7 kB 476.3 kB/s eta 0:00:01
   ----------- -------------------------- 122.9/413.7 kB 481.4 kB/s eta 0:00:01
   ------------- ------------------------ 143.4/413.7 kB 502.3 kB/s eta 0:00:01
   --------------- ---------------------- 174.1/413.7 kB 499.5 kB/s eta 0:00:01
   ------------------ ------------------- 204.8/413.7 kB 541.9 kB/s eta 0:00:01
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 4.25.8 which is incompatible.
